# Merge TRIPOD References

This notebook merges the reference files from the different instances of TRIPOD and TRIPOD+AI in PubMed

In [1]:
import pandas as pd
import glob
import os

REFERENCES_FOLDER = "../data/00_references"
TRIPOD_FOLDER = f"{REFERENCES_FOLDER}/TRIPOD"
TRIPOD_AI_FOLDER = f"{REFERENCES_FOLDER}/TRIPOD_AI"

# Papers referencing TRIPOD Statement

In [2]:
tripod_csv_files = glob.glob(os.path.join(TRIPOD_FOLDER, "*_11_Aug_25.csv"))

In [3]:
dfs_TRIPOD = {}

for file in tripod_csv_files:
    # Extract the integer prefix from filename
    base = os.path.basename(file)
    int_name = int(base.split("_")[0])  # Convert to int
    
    # Read CSV and remove the first row
    df = pd.read_csv(file)
    df = df.iloc[1:].reset_index(drop=True)
    
    # Store in dict
    dfs_TRIPOD[int_name] = df

In [4]:
len(dfs_TRIPOD.keys())

12

In [5]:
# Step 1: Concatenate all dataframes into one
TRIPOD_combined_df = pd.concat(dfs_TRIPOD.values(), ignore_index=True)
print(f"Initial combined number of references: {TRIPOD_combined_df.shape[0]}")

# Step 2: Remove duplicates based on PMID
before = TRIPOD_combined_df.shape[0]
TRIPOD_combined_df = TRIPOD_combined_df.drop_duplicates(subset=["PMID"], keep="first")
after = TRIPOD_combined_df.shape[0]
print(f"Removed {before - after} duplicate PMIDs. New number of references: {TRIPOD_combined_df.shape[0]}")

# Step 3: Remove rows where PMID equals one of the TRIPOD_PMIDs
TRIPOD_PMIDs = set(dfs_TRIPOD.keys())  # the integer prefixes from file names
before = TRIPOD_combined_df.shape[0]
mask = TRIPOD_combined_df["PMID"].isin(TRIPOD_PMIDs)
rows_to_drop = TRIPOD_combined_df[mask]
print(f"Removed {len(rows_to_drop)} rows that were one of the TRIPOD statements.")
TRIPOD_combined_df = TRIPOD_combined_df[~mask]
after = TRIPOD_combined_df.shape[0]
print(f"Final number of references: {TRIPOD_combined_df.shape[0]}")

# Reset index for cleanliness
TRIPOD_combined_df = TRIPOD_combined_df.reset_index(drop=True)

Initial combined number of references: 6762
Removed 453 duplicate PMIDs. New number of references: 6309
Removed 3 rows that were one of the TRIPOD statements.
Final number of references: 6306


In [6]:
TRIPOD_combined_df.to_csv(f"{TRIPOD_FOLDER}/TRIPOD_combined_references.csv", index=False)

# Papers referencing TRIPOD+AI Statement

In [7]:
tripod_ai_csv_files = glob.glob(os.path.join(TRIPOD_AI_FOLDER, "*_11_Aug_25.csv"))

In [8]:
dfs_TRIPOD_AI = {}

for file in tripod_ai_csv_files:
    # Extract the integer prefix from filename
    base = os.path.basename(file)
    int_name = int(base.split("_")[0])  # Convert to int
    
    # Read CSV and remove the first row
    df = pd.read_csv(file)
    df = df.iloc[1:].reset_index(drop=True)
    
    # Store in dict
    dfs_TRIPOD_AI[int_name] = df

In [9]:
len(dfs_TRIPOD_AI.keys())

2

In [10]:
# Step 1: Concatenate all dataframes into one
TRIPOD_AI_combined_df = pd.concat(dfs_TRIPOD_AI.values(), ignore_index=True)
print(f"Initial combined number of references: {TRIPOD_AI_combined_df.shape[0]}")

# Step 2: Remove duplicates based on PMID
before = TRIPOD_AI_combined_df.shape[0]
TRIPOD_AI_combined_df = TRIPOD_AI_combined_df.drop_duplicates(subset=["PMID"], keep="first")
after = TRIPOD_AI_combined_df.shape[0]
print(f"Removed {before - after} duplicate PMIDs. New number of references: {TRIPOD_AI_combined_df.shape[0]}")

# Step 3: Remove rows where PMID equals one of the TRIPOD_AI_PMIDs
TRIPOD_AI_PMIDs = set(dfs_TRIPOD_AI.keys())  # the integer prefixes from file names
before = TRIPOD_AI_combined_df.shape[0]
mask = TRIPOD_AI_combined_df["PMID"].isin(TRIPOD_AI_PMIDs)
rows_to_drop = TRIPOD_AI_combined_df[mask]
print(f"Removed {len(rows_to_drop)} rows that were one of the TRIPOD+AI statements.")
TRIPOD_AI_combined_df = TRIPOD_AI_combined_df[~mask]
after = TRIPOD_AI_combined_df.shape[0]
print(f"Final number of references: {TRIPOD_AI_combined_df.shape[0]}")

# Reset index for cleanliness
TRIPOD_AI_combined_df = TRIPOD_AI_combined_df.reset_index(drop=True)

Initial combined number of references: 411
Removed 1 duplicate PMIDs. New number of references: 410
Removed 0 rows that were one of the TRIPOD+AI statements.
Final number of references: 410


In [11]:
TRIPOD_AI_combined_df.to_csv(f"{TRIPOD_AI_FOLDER}/TRIPOD_AI_combined_references.csv", index=False)

# Merge TRIPOD and TRIPOD+AI references

In [12]:
df_combined = pd.concat([TRIPOD_combined_df, TRIPOD_AI_combined_df], ignore_index=True)

In [13]:
exclude_PMIDs = TRIPOD_PMIDs.union(TRIPOD_AI_PMIDs)

In [14]:
df_TRIPOD = pd.read_csv(f"{REFERENCES_FOLDER}/TRIPOD/TRIPOD_combined_references.csv")
df_TRIPOD_AI = pd.read_csv(f"{REFERENCES_FOLDER}/TRIPOD_AI/TRIPOD_AI_combined_references.csv")

In [15]:
df_combined = pd.concat([df_TRIPOD, df_TRIPOD_AI], ignore_index=True)

In [16]:
# Some papers appear twice if they cite both TRIPOD and TRIPOD+AI, so we remove them
# here. They are only removed if they are exact duplicates across all columns.

df_combined_dedup = df_combined.drop_duplicates().reset_index(drop=True)

In [17]:
# Checking for uniqueness to ensure there were no rows sharing the same PMID but with different data

assert df_combined_dedup["PMID"].is_unique, "PMID is not unique"

In [18]:
df_combined_dedup.to_parquet(f"{REFERENCES_FOLDER}/merged_references.parquet.brotli", index=False, compression="brotli")